# Lecture 6: Feature Engineering

Regression takes numbers. Real data has categories, text, and nonlinear relationships. Here's how to bridge that gap — from hand-crafted features to AI-generated ones.

In [Lecture 5](lec05-regression.qmd), we fit a regression predicting Airbnb prices from bedrooms and bathrooms. But we ignored most of the information in the dataset: borough, room type, listing descriptions. Today we learn **feature engineering** — the art of transforming raw data into inputs a model can use. We'll start with classical techniques (dummies, polynomials, interactions), meet decision trees that engineer features automatically, and end with a modern trick: using an LLM to extract structured features from unstructured text.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

## The data: Airbnb NYC listings

In [ ]:
DATA_DIR = 'https://raw.githubusercontent.com/stanford-mse-125/book/main/data'
listings = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False)

cols = ['id', 'name', 'description', 'price', 'bedrooms', 'bathrooms', 'room_type',
        'neighbourhood_group_cleansed', 'accommodates', 'number_of_reviews']
df = listings[cols].dropna(subset=['price', 'bedrooms', 'bathrooms']).copy()
df = df.rename(columns={'neighbourhood_group_cleansed': 'borough'})

# Clean price column: strip "$" and commas, convert to float
df['price'] = df['price'].astype(str).str.replace('[$,]', '', regex=True).astype(float)

# Focus on reasonable prices (filter out near-free and luxury outliers)
df = df[df['price'].between(10, 500)].reset_index(drop=True)
y = df['price']

print(f"Working with {len(df):,} listings")
df[['name', 'price', 'bedrooms', 'bathrooms', 'room_type', 'borough']].head()

## One-hot encoding

Regression needs numbers. But two of our most informative features — `borough` and `room_type` — are strings:

In [ ]:
print("borough values:", df['borough'].unique())
print("room_type values:", df['room_type'].unique())

We can't feed "Manhattan" or "Private room" into a regression equation. The solution is **one-hot encoding**: turn each category into a binary (0/1) column. A listing in Manhattan gets a 1 in the `borough_Manhattan` column and 0s elsewhere.

:::{.callout-important}
## Definition: Reference level
We use `drop_first=True` to avoid perfect collinearity. Why? If a listing isn't in any of the other boroughs, it *must* be in the dropped category (the **reference level**). Including all dummies would make the columns sum to the intercept column — a linearly dependent column space (recall the column space from [Lecture 4](lec04-linear-algebra.qmd)). The dropped category becomes the baseline that all other coefficients are measured against.
:::

In [ ]:
# One-hot encode categorical features
cat_features = pd.get_dummies(df[['room_type', 'borough']], drop_first=True)
num_features = df[['bedrooms', 'bathrooms']]
X_base = pd.concat([num_features, cat_features], axis=1)

# See what one-hot encoding produced
print("Encoded columns:")
print(cat_features.columns.tolist())
print(f"\nX shape: {X_base.shape}")
cat_features.head(3)

Each dummy column shifts predictions for that category. Let's fit a regression and see.

In [ ]:
model_base = LinearRegression().fit(X_base, y)
r2_base = model_base.score(X_base, y)

# Build coefficient table
feature_names = list(X_base.columns)
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': model_base.coef_
}).sort_values('coefficient', key=abs, ascending=False)
coef_df['coefficient'] = coef_df['coefficient'].round(2)
print(f"R² = {r2_base:.3f}\n")
print(coef_df.to_string(index=False))
print(f"\nIntercept (baseline = Bronx, Entire home/apt): {model_base.intercept_:.2f}")

How to read this: the coefficient on `borough_Manhattan` tells you how much more (or less) a Manhattan listing costs compared to the baseline category (Bronx), holding all else equal. Shared rooms are cheaper; Manhattan is pricier.

In [ ]:
# Bar chart of coefficients
fig, ax = plt.subplots(figsize=(10, 5))
coef_sorted = coef_df.sort_values('coefficient')
colors = ['#d62728' if c < 0 else '#2171b5' for c in coef_sorted['coefficient']]
ax.barh(coef_sorted['feature'], coef_sorted['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='gray', linewidth=1)
ax.set_xlabel('Coefficient (effect on price, $)')
ax.set_title('Regression coefficients: bedrooms + bathrooms + room type + borough')
plt.tight_layout()
plt.show()

## Polynomial features

What if the relationship isn't linear in the original features? Maybe the jump from 1 to 2 bedrooms matters more than from 4 to 5. We can capture this by adding **polynomial features**: $x$, $x^2$, $x^3$.

The key insight: the model is still **linear in parameters** — we're just giving it richer inputs.

$$\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2$$

This is still a linear regression! The model is nonlinear in $x$ but linear in the parameters $\beta$. That means we can still use least squares to fit it — the optimization is identical.

In [ ]:
X_bedrooms = df[['bedrooms']]

results = []
for degree in [1, 2, 3, 4]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly.fit_transform(X_bedrooms)
    m = LinearRegression().fit(X_poly, y)
    r2 = m.score(X_poly, y)
    results.append({'degree': degree, 'n_features': X_poly.shape[1], 'R²': round(r2, 4)})
    print(f"Degree {degree}: R² = {r2:.4f}  ({X_poly.shape[1]} features)")

print("\nDiminishing returns — polynomials on bedrooms alone aren't the answer.")
print("The real gains come from using the other information we have.")

In [ ]:
# Visualize polynomial fits
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: scatter with degree-1 and degree-3 fits
ax = axes[0]
ax.scatter(df['bedrooms'], y, alpha=0.05, s=5, color='gray')
x_line = np.linspace(0, df['bedrooms'].max(), 200).reshape(-1, 1)

for degree, color, ls in [(1, '#2171b5', '-'), (3, '#d62728', '--')]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_p = poly.fit_transform(X_bedrooms)
    m = LinearRegression().fit(X_p, y)
    y_line = m.predict(poly.transform(x_line))
    ax.plot(x_line, y_line, color=color, linewidth=2.5, linestyle=ls,
            label=f'Degree {degree} (R²={m.score(X_p, y):.3f})')

ax.set_xlabel('Bedrooms')
ax.set_ylabel('Price ($)')
ax.set_title('Polynomial fits: bedrooms → price')
ax.legend()

# Right: R² by degree
ax = axes[1]
degrees = [r['degree'] for r in results]
r2s = [r['R²'] for r in results]
ax.plot(degrees, r2s, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Polynomial degree')
ax.set_ylabel('R²')
ax.set_title('Diminishing returns from polynomial features')
ax.set_xticks(degrees)

plt.tight_layout()
plt.show()

:::{.callout-tip}
## Think About It
Why do polynomial features on bedrooms alone give such small improvements? What other sources of variation in price are we ignoring?
:::

## Interaction terms

Is an extra bedroom worth the same in Manhattan and the Bronx? So far our model assumes yes — there's one `bedrooms` coefficient that applies everywhere. An **interaction term** lets the effect of one feature depend on another:

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{bedrooms} + \beta_2 \cdot \mathbb{1}_{\text{Manhattan}} + \beta_3 \cdot (\text{bedrooms} \times \mathbb{1}_{\text{Manhattan}}) + \ldots$$

The interaction coefficient $\beta_3$ captures how much *more* (or less) a bedroom is worth in Manhattan compared to the reference borough.

In [ ]:
# Create interaction terms: bedrooms × each borough dummy
borough_dummies = pd.get_dummies(df['borough'], drop_first=True)
interactions = borough_dummies.multiply(df['bedrooms'], axis=0)
interactions.columns = [f'bedrooms_x_{col}' for col in interactions.columns]

# Full model with interactions
X_interact = pd.concat([X_base, interactions], axis=1)
model_interact = LinearRegression().fit(X_interact, y)
r2_interact = model_interact.score(X_interact, y)

print(f"Without interactions: R² = {r2_base:.3f}  ({X_base.shape[1]} features)")
print(f"With interactions:    R² = {r2_interact:.3f}  ({X_interact.shape[1]} features)")
print(f"Improvement: {r2_interact - r2_base:.4f}")

In [ ]:
# Visualize: different slopes per borough
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette('Set2', n_colors=df['borough'].nunique())

for color, (borough, group) in zip(colors, df.groupby('borough')):
    ax.scatter(group['bedrooms'], group['price'], alpha=0.03, s=5, color=color)
    # Fit within-borough regression line
    if len(group) > 50:
        mb = LinearRegression().fit(group[['bedrooms']], group['price'])
        x_range = np.array([[0], [group['bedrooms'].max()]])
        ax.plot(x_range, mb.predict(x_range), color=color, linewidth=2.5,
                label=f'{borough} (${mb.coef_[0]:.0f}/bedroom)')

ax.set_xlabel('Bedrooms')
ax.set_ylabel('Price ($)')
ax.set_title('Price per bedroom varies by borough')
ax.legend(loc='upper left', fontsize=9)
ax.set_xlim(-0.5, 6.5)
plt.tight_layout()
plt.show()

The slopes are quite different! An extra bedroom in Manhattan is worth much more than in the Bronx or Staten Island. Without interaction terms, our model uses a single average slope that may not represent *any* borough well.

In [ ]:
# Show R² progression
fig, ax = plt.subplots(figsize=(8, 4))
models = ['bedrooms only', '+ bath + room + borough', '+ interactions']
r2_list = [
    LinearRegression().fit(df[['bedrooms']], y).score(df[['bedrooms']], y),
    r2_base,
    r2_interact
]
bars = ax.bar(models, r2_list, color=['#aec7e8', '#6baed6', '#2171b5'], edgecolor='white')
for bar, r2_val in zip(bars, r2_list):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'R²={r2_val:.3f}', ha='center', fontsize=11)
ax.set_ylabel('R²')
ax.set_title('Feature engineering progressively improves the model')
ax.set_ylim(0, 0.6)
plt.tight_layout()
plt.show()

**Feature engineering is where domain knowledge meets statistics.** Knowing that location affects the value of space is domain knowledge; encoding it as an interaction term is statistics.

As we add more features, R^2 can only go up — even if the new features are noise. **Adjusted R^2** corrects for this by penalizing model complexity: $R^2_{adj} = 1 - \frac{s^2_{\text{residuals}} / (n-k-1)}{s^2_{\text{outcome}} / (n-1)}$, where $k$ is the number of predictors. A feature that doesn't help prediction will increase $k$ without sufficiently reducing the residual variance, causing adjusted R^2 to drop. This makes it a useful guide for deciding which features to keep.

<!-- FLAG: IMS Ch 8 covers model selection strategies (forward selection, backward elimination, stepwise) and Ch 10 covers multicollinearity and VIF in detail. These are not included here. Consider whether multicollinearity / VIF deserves a section — the lecture mentions collinearity only in the context of drop_first. Stepwise selection could fit in a model-building lecture. -->

:::{.callout-note}
## George Box on models
"All models are wrong, but some are useful." — George Box
:::

## A different approach — decision trees

Engineering features by hand is creative but tedious. What if the model could find the right splits on its own?

A **decision tree** splits the feature space into regions and predicts the mean price in each region. Each internal node asks an if/then question about one feature: "Is bedrooms > 2?" or "Is borough = Manhattan?" The tree learns which questions to ask and where to split — no manual feature engineering needed.

In [ ]:
# Train/test split for fair comparison (we'll formalize this idea in Lecture 7)
X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.3, random_state=42
)

# Decision tree with limited depth to prevent overfitting
tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

tree_train_r2 = tree.score(X_train, y_train)
tree_test_r2 = tree.score(X_test, y_test)

# Compare to linear model with interactions
model_interact_split = LinearRegression().fit(
    X_interact.loc[X_train.index], y_train
)
lm_train_r2 = model_interact_split.score(X_interact.loc[X_train.index], y_train)
lm_test_r2 = r2_score(y_test, model_interact_split.predict(X_interact.loc[X_test.index]))

print(f"Decision Tree (depth=4):")
print(f"  Train R² = {tree_train_r2:.3f}  |  Test R² = {tree_test_r2:.3f}")
print(f"\nLinear model with interactions:")
print(f"  Train R² = {lm_train_r2:.3f}  |  Test R² = {lm_test_r2:.3f}")

In [ ]:
# Visualize the tree (using max_depth=3 for readability)
tree_viz = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_viz.fit(X_train, y_train)

fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(tree_viz, feature_names=feature_names, filled=True, fontsize=8,
          rounded=True, ax=ax, impurity=False, proportion=False)
ax.set_title('Decision tree for Airbnb price prediction (depth=3)', fontsize=14)
plt.tight_layout()
plt.show()

Each leaf shows the predicted price (the mean of training examples that land there) and how many samples it contains. The tree automatically discovers that room type and borough matter — no one-hot encoding intuition required.

:::{.callout-tip}
## Think About It
The tree picks its own split points. Look at the first split — does it match your intuition about what matters most for price?
:::

## Random forests as benchmark

One tree overfits. What about averaging many trees?

A **random forest** fits hundreds of decision trees, each on a random subset of the data and features, then averages their predictions. The randomness ensures the trees disagree on noise but agree on signal.

In [ ]:
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_r2 = rf.score(X_train, y_train)
rf_test_r2 = rf.score(X_test, y_test)

print("Model comparison (test R²):")
print(f"  Linear (with interactions):  {lm_test_r2:.3f}")
print(f"  Decision Tree (depth=4):     {tree_test_r2:.3f}")
print(f"  Random Forest (100 trees):   {rf_test_r2:.3f}")

The forest beats our hand-engineered linear model — and we didn't engineer a single feature. It handles nonlinearities, interactions, and complex splits automatically.

How does this work? We'll understand the key idea in [Lecture 7](lec07-trees-validation.qmd) (averaging reduces variance) and go deep on tree-based methods in Lecture 17.

## Classification with trees

So far we've predicted prices — a continuous number. What about predicting categories?

**Classification** assigns observations to discrete groups instead of predicting a number. The same tree-based ideas apply: instead of predicting the mean in each leaf, a classification tree predicts the majority class.

In [ ]:
# Create a binary target: is this listing above median price?
df['is_expensive'] = (df['price'] > df['price'].median()).astype(int)
y_class = df['is_expensive']

print(f"Median price: ${df['price'].median():.0f}")
print(f"Expensive (above median): {y_class.sum():,} listings")
print(f"Not expensive:            {(1 - y_class).sum():,} listings")

In [ ]:
# Split and fit a random forest classifier
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_base, y_class, test_size=0.3, random_state=42
)

rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_c, y_train_c)

train_acc = rf_clf.score(X_train_c, y_train_c)
test_acc = rf_clf.score(X_test_c, y_test_c)

print(f"Random Forest Classifier:")
print(f"  Train accuracy: {train_acc:.3f}")
print(f"  Test accuracy:  {test_acc:.3f}")
print(f"  (Baseline — always guess majority class: {max(y_class.mean(), 1-y_class.mean()):.3f})")

A random forest classifier uses majority vote instead of averaging: each tree votes for a class, and the forest picks the most popular answer. You now have a classification tool for your projects. We'll build a deeper, more statistical understanding of classification with logistic regression in Lecture 13.

## LLM-based featurization

What about unstructured data like text? Our Airbnb listings have names and descriptions that might contain useful information — mentions of luxury, cleanliness, neighborhood character — but a regression can't read English.

Let's look at a few examples:

In [ ]:
# Show 3 raw listings with names and descriptions
sample = df[df['description'].notna()].sample(3, random_state=42)
for _, row in sample.iterrows():
    print(f"Name: {row['name']}")
    desc = str(row['description'])[:200]  # Truncate for display
    print(f"Description: {desc}...")
    print(f"Price: ${row['price']:.0f}")
    print()

An **LLM** (large language model) can read each description and extract structured features — numbers and categories that a regression *can* use. This is a form of **weak supervision**: instead of hand-labeling thousands of listings, you write a prompt once and let the model do the labeling.

In [ ]:
# How these features were generated (see data/airbnb/generate_llm_features.py):
#
# from anthropic import Anthropic
# client = Anthropic()
# response = client.messages.create(
#     model="claude-haiku-4-5-20251001",  # Fast + cheap: ~$0.01 per 1000 listings
#     messages=[{"role": "user", "content": f"Extract from this Airbnb listing:\n{description}\n\n"
#                "Return JSON: mentions_luxury (bool), amenity_count (int), sentiment_score (0-1), "
#                "neighborhood_vibe (category), cleanliness_mentioned (bool)"}],
# )
#
# For projects: Claude Haiku is best for bulk extraction (fast, cheap).
# Use Claude Sonnet for complex reasoning tasks.
# Always validate: LLM features can be wrong. Cross-validate to check if they help.

# Load pre-computed LLM features
llm_features = pd.read_csv(f'{DATA_DIR}/airbnb/llm_features.csv')
print(f"LLM features shape: {llm_features.shape}")
print(f"Columns: {llm_features.columns.tolist()}")
llm_features.head()

In [ ]:
# Join LLM features to the main dataframe on listing id
df_llm = df.merge(llm_features, on='id', how='inner')
print(f"Listings with LLM features: {len(df_llm):,} (of {len(df):,} total)")

# Show the distribution of LLM-extracted features
llm_cols = [c for c in llm_features.columns if c != 'id']
print(f"\nLLM feature summary:")
print(df_llm[llm_cols].describe().round(2))

In [ ]:
# Build feature matrix with LLM features included
y_llm = df_llm['price']

# Numeric + categorical features (same as before, on the joined subset)
cat_llm = pd.get_dummies(df_llm[['room_type', 'borough']], drop_first=True)
num_llm = df_llm[['bedrooms', 'bathrooms']]

# Without LLM features
X_no_llm = pd.concat([num_llm, cat_llm], axis=1)

# With LLM features
llm_numeric = df_llm[llm_cols].apply(pd.to_numeric, errors='coerce').fillna(0)
X_with_llm = pd.concat([num_llm, cat_llm, llm_numeric], axis=1)

# Train/test split on the LLM subset
X_tr_no, X_te_no, y_tr_llm, y_te_llm = train_test_split(
    X_no_llm, y_llm, test_size=0.3, random_state=42
)
X_tr_with = X_with_llm.loc[X_tr_no.index]
X_te_with = X_with_llm.loc[X_te_no.index]

# Fit models with and without LLM features
m_no_llm = LinearRegression().fit(X_tr_no, y_tr_llm)
m_with_llm = LinearRegression().fit(X_tr_with, y_tr_llm)

r2_no = r2_score(y_te_llm, m_no_llm.predict(X_te_no))
r2_with = r2_score(y_te_llm, m_with_llm.predict(X_te_with))

print(f"Without LLM features: Test R² = {r2_no:.3f}  ({X_no_llm.shape[1]} features)")
print(f"With LLM features:    Test R² = {r2_with:.3f}  ({X_with_llm.shape[1]} features)")
print(f"Improvement:           {r2_with - r2_no:+.3f}")

In [ ]:
# Show which LLM features matter most
all_features_llm = list(X_with_llm.columns)
coef_llm_df = pd.DataFrame({
    'feature': all_features_llm,
    'coefficient': m_with_llm.coef_
})
# Highlight LLM features
coef_llm_df['source'] = ['LLM' if f in llm_cols else 'original' for f in all_features_llm]
coef_llm_df = coef_llm_df.sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#d62728' if s == 'LLM' else '#2171b5' for s in coef_llm_df['source']]
ax.barh(range(len(coef_llm_df)), coef_llm_df['coefficient'], color=colors, edgecolor='white')
ax.set_yticks(range(len(coef_llm_df)))
ax.set_yticklabels(coef_llm_df['feature'], fontsize=9)
ax.axvline(0, color='gray', linewidth=1)
ax.set_xlabel('Coefficient (effect on price, $)')
ax.set_title('Regression coefficients: original features (blue) + LLM features (red)')
plt.tight_layout()
plt.show()

:::{.callout-warning}
## LLM features can be wrong
LLM featurization is a powerful modern technique: instead of building a natural language processing pipeline from scratch, you write a prompt and let the model extract structured information. The resulting features are imperfect — LLMs can hallucinate or misinterpret text — but cross-validation tells you whether they help. Always validate LLM-extracted features before trusting them in a model.
:::

## Key takeaways

- **One-hot encoding** turns categories into numbers. Each dummy shifts predictions for that group; `drop_first=True` avoids collinearity with the intercept.
- **Polynomial features** and **interaction terms** make linear models arbitrarily powerful. The mantra: **"linear in parameters, not in features."**
- **Decision trees** find splits automatically — no manual feature engineering needed. Each node asks an if/then question about one feature.
- **Random forests** average many trees to reduce overfitting. They provide a strong benchmark without any feature engineering.
- **Random forest classifiers** use majority vote for classification — a tool you can use immediately in your projects.
- **LLM featurization** extracts structured features from unstructured text. This is **weak supervision** at scale: one prompt replaces thousands of hand labels.
- Next: [Lecture 7](lec07-trees-validation.qmd) asks "how do we know these features aren't just overfitting?" — introducing train/test splits, cross-validation, and the bias-variance tradeoff.

## Study guide

### Key ideas

- **Feature engineering** — creating new input variables from raw data to improve a model's predictions.
- **One-hot encoding** converts a categorical variable with $k$ categories into $k-1$ binary (0/1) columns (the dropped category is the **reference level**). Each dummy coefficient measures a shift relative to that baseline.
- **Polynomial features** — powers of a variable ($x^2, x^3, \ldots$) that let a linear model capture curved relationships. The model is still linear in $\beta$, not in $x$.
- **Interaction terms** — products of two features (e.g., bedrooms $\times$ $\mathbb{1}_{\text{Manhattan}}$) that let one feature's effect depend on another.
- **Adjusted R^2** penalizes model complexity, preventing R^2 from rewarding useless features. Use it to compare models with different numbers of predictors.
- **Decision trees** recursively split the feature space into regions, predicting the mean (regression) or majority class (classification) in each region — no manual feature engineering needed.
- **Random forests** average many decision trees trained on random subsets of data and features, reducing overfitting. More trees never hurts.
- **Weak supervision** — using noisy, programmatic labels (e.g., from an LLM) instead of expensive hand labels. LLM featurization converts unstructured text into structured features using a single prompt.

### Computational tools

- `pd.get_dummies(df, drop_first=True)` — one-hot encode categorical columns, dropping one to avoid collinearity
- `PolynomialFeatures(degree=d)` — generates polynomial features up to degree $d$; still linear in parameters
- `DecisionTreeRegressor(max_depth=k)` — fits a regression tree with at most $k$ levels of splits
- `RandomForestRegressor(n_estimators=100)` — fits 100 trees and averages predictions
- `RandomForestClassifier(n_estimators=100)` — fits 100 trees and uses majority vote for classification
- `train_test_split(X, y, test_size=0.3, random_state=42)` — splits data for honest evaluation

### For the quiz

- Know what one-hot encoding produces and why `drop_first=True` is needed.
- Be able to write the formula for a model with polynomial or interaction features and explain why it's still "linear regression."
- Understand at a high level how a decision tree makes predictions (recursive splits, predict mean in each leaf).
- Know the difference between regression (predict a number) and classification (predict a category).
- Be able to explain what LLM featurization does and why it's called weak supervision.